In [1]:
import gymnasium as gym
import env  # registers my-four-room-v0
import numpy as np
import wandb
import matplotlib.pyplot as plt

from morl_baselines.multi_policy.pareto_q_learning.pql import PQL
from wrappers import CombineWrapper, Convert2DStateWrapper
from utils import visualize_front_general

In [2]:
# Experiment config
PROJECT = "MORL-Baselines"
RUN_NAME = "pql-my-four-room-interpretations-per-value-weight"

TOTAL_TIMESTEPS = int(1e5)
SEED = 42
N_WEIGHTS = 10
MAX_EPISODE_STEPS = 300

# After CombineWrapper, rewards are [triangle, circle]
REF_POINT = np.array([0.0, 0.0], dtype=np.float32)
VALUE_WEIGHTS = np.linspace(0.0, 1.0, N_WEIGHTS)

In [ ]:
all_fronts = {}

with wandb.init(project=PROJECT, name=RUN_NAME, config={
    "total_timesteps": TOTAL_TIMESTEPS,
    "seed": SEED,
    "n_weights": N_WEIGHTS,
}):
    for i, w_blue in enumerate(VALUE_WEIGHTS):
        w_red = 1.0 - w_blue
        weight_vec = np.array([w_blue, w_red], dtype=np.float32)

        train_env = CombineWrapper(
            Convert2DStateWrapper(
                gym.wrappers.TimeLimit(gym.make("my-four-room-v0"), max_episode_steps=MAX_EPISODE_STEPS)
            ),
            weight_vec,
        )
        eval_env = CombineWrapper(
            Convert2DStateWrapper(
                gym.wrappers.TimeLimit(gym.make("my-four-room-v0"), max_episode_steps=MAX_EPISODE_STEPS)
            ),
            weight_vec,
        )

        # Sanity: wrapper combines 4 original objectives into 2 interpretation objectives
        assert train_env.unwrapped.reward_space.shape == (2,), (
            f"Expected wrapped reward shape (2,), got {train_env.unwrapped.reward_space.shape}"
        )
        # Sanity: PQL gets compact tabular-friendly state shape
        assert train_env.observation_space.shape == (2,), (
            f"Expected converted observation shape (2,), got {train_env.observation_space.shape}"
        )

        agent = PQL(
            env=train_env,
            ref_point=REF_POINT,
            gamma=0.99,
            initial_epsilon=1.0,
            epsilon_decay_steps=TOTAL_TIMESTEPS,
            final_epsilon=0.05,
            seed=SEED,
            log=False,
        )

        # Heartbeat log before potentially long training call
        wandb.log({
            "outer_idx": i,
            "value_w_blue": float(w_blue),
            "value_w_red": float(w_red),
            "phase": "train_start",
        })

        pareto_front = agent.train(
            total_timesteps=TOTAL_TIMESTEPS,
            eval_env=eval_env,
            ref_point=REF_POINT,
            log_every=max(1, TOTAL_TIMESTEPS // 10),
            action_eval="hypervolume",
        )

        front_arr = np.array(list(pareto_front), dtype=np.float32)
        all_fronts[float(w_blue)] = front_arr

        # Per-weight logging in one shared run
        wandb.log({
            "outer_idx": i,
            "value_w_blue": float(w_blue),
            "value_w_red": float(w_red),
            "front_size": int(len(front_arr)),
        })

        visualize_front_general(front_arr, columns=["triangle", "circle"])

        print(
            f"[{i + 1}/{N_WEIGHTS}] value weights (blue={w_blue:.3f}, red={w_red:.3f}) -> "
            f"front size={len(front_arr)}"
        )

        train_env.close()
        eval_env.close()

    wandb.log({"n_fronts_collected": len(all_fronts)})

    # Aggregate visualization logged in the same run
    fig, ax = plt.subplots(figsize=(8, 6))
    for w_blue, front in all_fronts.items():
        if len(front) == 0:
            continue
        ax.scatter(front[:, 0], front[:, 1], alpha=0.7, label=f"w_blue={w_blue:.2f}")

    ax.set_xlabel("triangle interpretation")
    ax.set_ylabel("circle interpretation")
    ax.set_title("Pareto fronts over interpretations per value scalarization weight")
    ax.legend(loc="best", fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    wandb.log({"all_fronts_overlay": wandb.Image(fig)})
    plt.close(fig)

print("Done. Collected fronts for", len(all_fronts), "value weights.")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: kristofs (kristofs-ai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


c:\Users\skakr\Documents\Programming\md-morl\.venv\Lib\site-packages\gymnasium\utils\passive_env_checker.py:245: UserWarning: WARN: The reward returned by `step()` must be a float, int, np.integer or np.floating, actual type: <class 'numpy.ndarray'>
  logger.warn(
Traceback (most recent call last):
  File "C:\Users\skakr\AppData\Local\Temp\ipykernel_6124\56146960.py", line 35, in <module>
    pareto_front = agent.train(
                   ^^^^^^^^^^^^
  File "c:\Users\skakr\Documents\Programming\md-morl\.venv\Lib\site-packages\morl_baselines\multi_policy\pareto_q_learning\pql.py", line 256, in train
    self.non_dominated[state][action] = self.calc_non_dominated(next_state)
                                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\skakr\Documents\Programming\md-morl\.venv\Lib\site-packages\morl_baselines\multi_policy\pareto_q_learning\pql.py", line 196, in calc_non_dominated
    non_dominated = get_non_dominated(candidates)
                    ^^^^^^^

KeyboardInterrupt: 

In [ ]:
# Optional: inspect one front (example for w_blue ~= 0.5)
example_key = min(all_fronts.keys(), key=lambda x: abs(x - 0.5))
example_front = all_fronts[example_key]

print(f"Example key: w_blue={example_key:.3f}, points={len(example_front)}")
print(example_front[:10])